In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

print("✅ Imports OK")

✅ Imports OK


In [9]:
df         = pd.read_csv(r'C:\Users\Piwi\Documents\VS\WorldCup2026\data\results.csv')
df_ranking = pd.read_csv(r'C:\Users\Piwi\Documents\VS\WorldCup2026\data\fifa_ranking-2024-06-20.csv')

In [10]:
# Correspondance nom équipe → nom dans results.csv
equipes_48 = {
    'France': 'France', 'Argentina': 'Argentina', 'Brazil': 'Brazil',
    'England': 'England', 'Spain': 'Spain', 'Portugal': 'Portugal',
    'Netherlands': 'Netherlands', 'Belgium': 'Belgium', 'Germany': 'Germany',
    'Croatia': 'Croatia', 'Uruguay': 'Uruguay', 'Colombia': 'Colombia',
    'Mexico': 'Mexico', 'USA': 'United States', 'Canada': 'Canada',
    'Senegal': 'Senegal', 'Morocco': 'Morocco', 'Japan': 'Japan',
    'Korea Republic': 'South Korea', 'Australia': 'Australia',
    'IR Iran': 'Iran', 'Saudi Arabia': 'Saudi Arabia', 'Ecuador': 'Ecuador',
    'Paraguay': 'Paraguay', 'Switzerland': 'Switzerland', 'Austria': 'Austria',
    'Turkey': 'Turkey', 'Norway': 'Norway', 'Scotland': 'Scotland',
    'Czechia': 'Czech Republic', 'Bosnia': 'Bosnia-Herzegovina', 'Panama': 'Panama',
    'Haiti': 'Haiti', 'Egypt': 'Egypt', 'Algeria': 'Algeria',
    'Tunisia': 'Tunisia', "Cote d'Ivoire": 'Ivory Coast', 'Ghana': 'Ghana',
    'South Africa': 'South Africa', 'Congo DR': 'DR Congo', 'Jordan': 'Jordan',
    'New Zealand': 'New Zealand', 'Uzbekistan': 'Uzbekistan', 'Qatar': 'Qatar',
    'Curacao': 'Curacao', 'Cape Verde': 'Cape Verde',
    'Sweden': 'Sweden', 'Iraq': 'Iraq',
}

# Pays hôtes → avantage domicile
pays_hotes = ['USA', 'Canada', 'Mexico']

# Filtrer sur 10 dernières années
df['date'] = pd.to_datetime(df['date'])
df_recents = df[df['date'] >= '2014-01-01'].dropna(subset=['home_score', 'away_score'])
print(f"✅ Matchs depuis 2014 : {len(df_recents)}")

# Classement FIFA — rang le plus récent par équipe
df_ranking_clean = (
    df_ranking
    .sort_values('rank_date', ascending=False)
    .drop_duplicates(subset='country_full', keep='first')
    [['country_full', 'total_points']]
    .copy()
)
df_ranking_clean.columns = ['team_csv', 'fifa_points']

# Calcul stats par équipe
rows = []
for app_name, csv_name in equipes_48.items():
    home_m = df_recents[df_recents['home_team'] == csv_name]
    away_m = df_recents[df_recents['away_team'] == csv_name]
    total  = len(home_m) + len(away_m)

    if total == 0:
        print(f"⚠️ Aucun match trouvé : {app_name}")
        moy_buts_marques = moy_buts_encaisses = 1.5
        nb_victoires = 0
        taux_victoire = 0.33
        forme_10ans = 0.5
    else:
        # Buts marqués et encaissés
        moy_buts_marques   = (home_m['home_score'].sum() + away_m['away_score'].sum()) / total
        moy_buts_encaisses = (home_m['away_score'].sum() + away_m['home_score'].sum()) / total

        # Victoires
        v_home = len(home_m[home_m['home_score'] > home_m['away_score']])
        v_away = len(away_m[away_m['away_score'] > away_m['home_score']])
        nb_victoires  = v_home + v_away
        taux_victoire = nb_victoires / total

        # Forme 10 ans — V=3pts, N=1pt, D=0pt normalisé entre 0 et 1
        points = 0
        for _, r in home_m.iterrows():
            if r['home_score'] > r['away_score']:    points += 3
            elif r['home_score'] == r['away_score']: points += 1
        for _, r in away_m.iterrows():
            if r['away_score'] > r['home_score']:    points += 3
            elif r['away_score'] == r['home_score']: points += 1
        forme_10ans = points / (total * 3)

    # Points FIFA
    fifa_row = df_ranking_clean[df_ranking_clean['team_csv'] == csv_name]
    fifa_points = float(fifa_row['fifa_points'].values[0]) if len(fifa_row) > 0 else 1400.0

    # Rang FIFA
    ranking_sorted = df_ranking_clean.sort_values('fifa_points', ascending=False).reset_index(drop=True)
    rank_row = ranking_sorted[ranking_sorted['team_csv'] == csv_name]
    fifa_rank = int(rank_row.index[0]) + 1 if len(rank_row) > 0 else 50

    rows.append({
        'team':               app_name,
        'fifa_points':        round(fifa_points, 2),
        'fifa_rank':          fifa_rank,
        'nb_matchs':          total,
        'nb_victoires':       nb_victoires,
        'taux_victoire':      round(taux_victoire, 3),
        'moy_buts_marques':   round(moy_buts_marques, 3),
        'moy_buts_encaisses': round(moy_buts_encaisses, 3),
        'forme_10ans':        round(forme_10ans, 3),
        'avantage_domicile':  1 if app_name in pays_hotes else 0,
    })

# Créer et afficher le DataFrame
df_48 = pd.DataFrame(rows).sort_values('fifa_points', ascending=False).reset_index(drop=True)

print(f"\n✅ Dataset créé : {len(df_48)} équipes")
print(f"\n📊 Colonnes : {df_48.columns.tolist()}")
df_48

✅ Matchs depuis 2014 : 11739
⚠️ Aucun match trouvé : Bosnia
⚠️ Aucun match trouvé : Curacao

✅ Dataset créé : 48 équipes

📊 Colonnes : ['team', 'fifa_points', 'fifa_rank', 'nb_matchs', 'nb_victoires', 'taux_victoire', 'moy_buts_marques', 'moy_buts_encaisses', 'forme_10ans', 'avantage_domicile']


,team,fifa_points,fifa_rank,nb_matchs,nb_victoires,taux_victoire,moy_buts_marques,moy_buts_encaisses,forme_10ans,avantage_domicile
0,Argentina,1860.14,1,155,102,0.658,1.981,0.658,0.729,0
1,France,1837.47,2,159,106,0.667,2.119,0.849,0.732,0
2,Belgium,1797.98,3,149,99,0.664,2.416,0.866,0.725,0
3,Brazil,1791.85,4,151,98,0.649,2.033,0.675,0.720,0
4,England,1787.88,5,153,96,0.627,2.065,0.673,0.697,0
5,Portugal,1747.04,6,155,97,0.626,2.187,0.787,0.695,0
6,Netherlands,1746.66,7,143,81,0.566,2.154,1.035,0.639,0
7,Spain,1729.92,8,148,95,0.642,2.345,0.777,0.721,0
8,Croatia,1728.30,9,148,78,0.527,1.764,1.034,0.608,0
9,Colombia,1669.44,12,144,72,0.500,1.535,0.861,0.600,0


In [11]:
# ============================================================
# ÉTAPE 1 — Créer des matchs synthétiques pour l'entraînement
# ============================================================
# On génère toutes les paires possibles d'équipes
# et on utilise les matchs historiques pour labelliser

from itertools import combinations

matchs_train = []

for team_A, team_B in combinations(equipes_48.keys(), 2):
    csv_A = equipes_48[team_A]
    csv_B = equipes_48[team_B]

    # Chercher les matchs historiques entre ces deux équipes
    matchs_AB = df_recents[
        ((df_recents['home_team'] == csv_A) & (df_recents['away_team'] == csv_B)) |
        ((df_recents['home_team'] == csv_B) & (df_recents['away_team'] == csv_A))
    ]

    if len(matchs_AB) == 0:
        continue  # Pas d'historique → on skip

    for _, match in matchs_AB.iterrows():
        # Déterminer le résultat réel
        if match['home_team'] == csv_A:
            if match['home_score'] > match['away_score']:
                label = 1  # A gagne
            elif match['home_score'] < match['away_score']:
                label = 0  # B gagne
            else:
                label = 2  # Nul
        else:
            if match['away_score'] > match['home_score']:
                label = 1  # A gagne
            elif match['away_score'] < match['home_score']:
                label = 0  # B gagne
            else:
                label = 2  # Nul

        # Récupérer les stats de chaque équipe dans df_48
        stats_A = df_48[df_48['team'] == team_A].iloc[0]
        stats_B = df_48[df_48['team'] == team_B].iloc[0]

        matchs_train.append({
            'diff_fifa_points':        stats_A['fifa_points']        - stats_B['fifa_points'],
            'diff_taux_victoire':      stats_A['taux_victoire']      - stats_B['taux_victoire'],
            'diff_moy_buts_marques':   stats_A['moy_buts_marques']   - stats_B['moy_buts_marques'],
            'diff_moy_buts_encaisses': stats_A['moy_buts_encaisses'] - stats_B['moy_buts_encaisses'],
            'diff_forme_10ans':        stats_A['forme_10ans']        - stats_B['forme_10ans'],
            'avantage_domicile_A':     stats_A['avantage_domicile'],
            'label': label
        })

df_matchs = pd.DataFrame(matchs_train)
print(f"✅ Dataset d'entraînement : {len(df_matchs)} matchs")
print(df_matchs['label'].value_counts())

✅ Dataset d'entraînement : 1377 matchs
label
1    721
2    345
0    311
Name: count, dtype: int64


In [12]:
# ============================================================
# ÉTAPE 2 — Entraîner le modèle
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df_matchs.drop('label', axis=1)
y = df_matchs['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred, target_names=['B gagne', 'A gagne', 'Nul']))

Accuracy : 43.12%
              precision    recall  f1-score   support

     B gagne       0.18      0.22      0.20        54
     A gagne       0.56      0.63      0.59       147
         Nul       0.34      0.20      0.25        75

    accuracy                           0.43       276
   macro avg       0.36      0.35      0.35       276
weighted avg       0.42      0.43      0.42       276



In [17]:
# ============================================================
# ÉTAPE 3 — Prédire un match entre deux équipes
# ============================================================
def predire_match(team_A, team_B, df_48, model):
    stats_A = df_48[df_48['team'] == team_A].iloc[0]
    stats_B = df_48[df_48['team'] == team_B].iloc[0]

    features = pd.DataFrame([{
        'diff_fifa_points':        stats_A['fifa_points']        - stats_B['fifa_points'],
        'diff_taux_victoire':      stats_A['taux_victoire']      - stats_B['taux_victoire'],
        'diff_moy_buts_marques':   stats_A['moy_buts_marques']   - stats_B['moy_buts_marques'],
        'diff_moy_buts_encaisses': stats_A['moy_buts_encaisses'] - stats_B['moy_buts_encaisses'],
        'diff_forme_10ans':        stats_A['forme_10ans']        - stats_B['forme_10ans'],
        'avantage_domicile_A':     stats_A['avantage_domicile'],
    }])

    proba = model.predict_proba(features)[0]
    classes = model.classes_  # [0, 1, 2] → [B gagne, A gagne, Nul]

    print(f"\n🏆 {team_A} vs {team_B}")
    for i, c in enumerate(classes):
        label = ['B gagne', 'A gagne', 'Nul'][c]
        print(f"  {label} : {proba[i]:.1%}")

# Exemple
predire_match('France', 'Brazil', df_48, model)
predire_match('Morocco', 'USA', df_48, model)


🏆 France vs Brazil
  B gagne : 48.3%
  A gagne : 49.5%
  Nul : 2.1%

🏆 Morocco vs USA
  B gagne : 9.8%
  A gagne : 73.0%
  Nul : 17.2%
